In [ ]:
import sys
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
import secrets
import os

from AES_CTR.AES_Lib import AES_Software
from ml_kem_driver import MLKem768

print("Libraries imported successfully!")

In [ ]:
BITSTREAM_DIR = "/root/jupyter_notebooks/verilog_ML_KEM/bitstream"

bitfile = os.environ.get("ML_KEM_BIT", os.path.join(BITSTREAM_DIR, "ml_kem_bd.bit"))

kem = MLKem768(bitfile)

In [ ]:
cap = cv2.VideoCapture(0)

ret, frame = cap.read()
if ret:
    # Resize nhỏ lại để demo chạy cho nhanh (ví dụ 320x240)
    frame = cv2.resize(frame, (320, 240))
    # Convert sang RGB để hiển thị đúng màu trên matplotlib
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    plt.imshow(frame_rgb)
    plt.title("Original Image (Input)")
    plt.show()
    
    print(f"Image Shape: {frame_rgb.shape}")
    print(f"Data Type: {frame_rgb.dtype}")
else:
    print("Error: Could not read from camera.")
    cap.release()

In [ ]:
d = secrets.token_bytes(32)
z = secrets.token_bytes(32)
m = secrets.token_bytes(32)
pk, sk, c1 = kem.keygen(d, z)
ct, ss1, c2 = kem.encaps(pk, m)
ss2, c3 = kem.decaps(sk, ct)
assert ss1 == ss2
print("SUCCESS: Shared secrets match!")

In [ ]:
print("--- START AES ENCRYPTION ---")

# Khởi tạo bộ mã hóa AES
my_aes_encrypt = AES_Software(ss1)

# Thực hiện mã hóa ảnh
encrypted_data = my_aes_encrypt.encrypt_image(frame_rgb)

# Hiển thị dữ liệu mã hóa (Sẽ trông như nhiễu)
# Reshape tạm để hiển thị dạng ảnh noise
encrypted_view = np.frombuffer(encrypted_data, dtype=np.uint8).reshape(frame_rgb.shape)
plt.imshow(encrypted_view)
plt.title("Encrypted Image (Ciphertext)")
plt.show()

In [ ]:
print("--- START AES DECRYPTION ---")

# Khởi tạo bộ mã hóa AES
my_aes_decrypt = AES_Software(ss2)

# Thực hiện giải mã
decrypted_img = my_aes_decrypt.decrypt_to_image(encrypted_data, frame_rgb.shape, frame_rgb.dtype)

# Hiển thị ảnh sau giải mã
plt.imshow(decrypted_img)
plt.title("Decrypted Image (Recovered)")
plt.show()

# Kiểm tra dữ liệu gốc và dữ liệu giải mã
if np.array_equal(frame_rgb, decrypted_img):
    print("SYSTEM VERIFIED: Decrypted image matches original exactly.")
else:
    print("ERROR: Data mismatch.")